# **implement of the Fuel cell Hybrid Electric vehicle EMS via Soft Actor-critic**

In [ ]:
# define the saved directory 
dir = r'\...\...\folder'

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import scipy.io as scio
from itertools import accumulate
from scipy.interpolate import interp1d, interp2d, RectBivariateSpline
import random
import copy
from collections import namedtuple, deque
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, MultivariateNormal
from torch.nn.utils import clip_grad_norm_
import torch.optim as optim

In [ ]:
class OneDimensionalLookupTable:
  def __init__(self, x_values, y_values):
    self.x_values = np.array(x_values)
    self.y_values = np.array(y_values)

  def lookup(self, input_value):
    output_value = np.interp(input_value, self.x_values, self.y_values)
    return output_value
  

def merge_lists(list1, list2):
    merged_list = []
    max_length = max(len(list1), len(list2))

    for i in range(max_length):
        if i < len(list1):
            merged_list.append(list1[i])
        if i < len(list2):
            merged_list.append(list2[i])

    return merged_list

# **Mirai Fuel cell hybrid model**

In [ ]:
class Mirai_model_V2():
  def __init__(self, P_max_fc = 114.5, Q_bat = 1.6):
      self.P_max_fc = P_max_fc
      self.Q_bat = Q_bat
      self.Q_nom = (self.Q_bat*1000*3600)/245
      self.mass_vehicle = 1793.45 + self.P_max_fc/1.6 + self.Q_bat*1000/24.5
      self.g=9.81
      self.c_d=0.29
      self.A_f=2.23
      self.rho=1.2
      self.R_wheel=0.316
      self.f_r=0.01
      self.J_wheel=0.32
      self.GR=9.09
      self.eta_gb=0.98
      self.J_em=0.008
      self.P_axu=440
      self.time_step = 1
      self.data_EM_speed = list(reversed([13127.7, 12787.2, 12106.4, 11595.7, 10851.1, 9978.72, 9063.83, 8127.66, 7553.19, 
                            6638.3, 5553.19, 4574.47, 4297.87, 3936.17, 3617.02, 3404.26, 3234.04, 2936.17, 2404.26, 1744.68, 1127.66, 510.638, 106.383, 0]))

      self.data_max_EM_torqe = list(reversed([80.893, 82.3665, 88.3066, 92.7616, 100.194, 109.114, 119.528, 132.934, 143.371,
                                161.268, 195.616, 235.957, 252.401, 274.826, 297.253, 319.687, 333.145, 335, 335, 335, 335, 335, 335, 335]))
      self.data_max_EM_power = []
      self.data_speed_power = []
      for i in range(len(self.data_max_EM_torqe)):
          self.data_max_EM_power.append(self.data_max_EM_torqe[i] * self.data_EM_speed[i] * 6.28/60 *0.001)
          self.data_speed_power.append(self.data_EM_speed[i] * 6.28/60 /self.GR *self.R_wheel)
      
      self.lookup_table_max_EM_power = OneDimensionalLookupTable(self.data_speed_power, self.data_max_EM_power)
      self.lookup_table_max_EM_torqe = OneDimensionalLookupTable(self.data_EM_speed, self.data_max_EM_torqe)

      self.x_values = np.array([0, 1350, 2700, 4050, 5400, 6750, 8100, 9450, 10800, 12150, 13500])
      self.y_values = np.array([-335, -301.5, -268, -234.5, -201, -167.5, -134, -100.5, -67, -33.5, 0, 33.5, 67, 100.5, 134, 167.5, 201, 234.5, 268, 301.5, 335])
      self.z_values = np.array([
            [0.7, 0.78, 0.85, 0.86, 0.81, 0.82, 0.79, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.78, 0.86, 0.87, 0.82, 0.82, 0.79, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.79, 0.86, 0.88, 0.85, 0.82, 0.79, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.8, 0.86, 0.89, 0.87, 0.82, 0.78, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.81, 0.87, 0.9, 0.88, 0.85, 0.79, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.82, 0.88, 0.9, 0.9, 0.87, 0.82, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.82, 0.87, 0.9, 0.91, 0.9, 0.86, 0.8, 0.78, 0.78, 0.78],
            [0.7, 0.82, 0.86, 0.9, 0.91, 0.91, 0.9, 0.88, 0.8, 0.78, 0.78],
            [0.7, 0.81, 0.85, 0.89, 0.91, 0.91, 0.91, 0.91, 0.9, 0.88, 0.8],
            [0.7, 0.77, 0.82, 0.87, 0.88, 0.89, 0.9, 0.91, 0.92, 0.92, 0.92],
            [0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7, 0.7],
            [0.7, 0.77, 0.82, 0.87, 0.88, 0.89, 0.9, 0.91, 0.92, 0.92, 0.92],
            [0.7, 0.81, 0.85, 0.89, 0.91, 0.91, 0.91, 0.91, 0.9, 0.88, 0.8],
            [0.7, 0.82, 0.86, 0.9, 0.91, 0.91, 0.9, 0.88, 0.8, 0.78, 0.78],
            [0.7, 0.82, 0.87, 0.9, 0.91, 0.9, 0.86, 0.8, 0.78, 0.78, 0.78],
            [0.7, 0.82, 0.88, 0.9, 0.9, 0.87, 0.82, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.81, 0.87, 0.9, 0.88, 0.85, 0.79, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.8, 0.86, 0.89, 0.87, 0.82, 0.78, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.79, 0.86, 0.88, 0.85, 0.82, 0.79, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.78, 0.86, 0.87, 0.82, 0.82, 0.79, 0.78, 0.78, 0.78, 0.78],
            [0.7, 0.78, 0.85, 0.86, 0.81, 0.82, 0.79, 0.78, 0.78, 0.78, 0.78],
        ])
      x_mesh, y_mesh = np.meshgrid(self.x_values, self.y_values)
      self.interp_function = RectBivariateSpline(self.x_values, self.y_values, self.z_values.T, kx=1, ky=1)

      self.data_M = [31630,21681,12934,15512]
      self.data_C =[0.5,2,6,10]
      self.lookup_table_M = OneDimensionalLookupTable(self.data_C, self.data_M)

      self.data_power = [-0.002408754,0.000263372,0.001621386,0.002984877,0.004375816,0.007075377,0.008482772,0.011182368,0.017879123,0.019292018,0.024630789,0.028660877,0.031382368,0.038035263,0.044710088,0.051362982,0.060655088,0.068594649,0.080454649,0.096223684,0.119800877,0.138094737,0.151158772,0.165504386,0.186409649,0.209921053,0.240003509,0.270128947,0.312120175,0.348910526,0.381757895,0.423787719,0.451345614,0.490675439,0.523396491,0.546941228,0.573075439,0.597917544,0.622742982,0.64889386,0.669809649,0.697318421,0.716964035,0.747144737,0.781295614,0.815429825,0.837714912,0.856046491,0.867757895,0.871657018]
      self.data_eta_stack = [0.566706,0.570041,0.573379,0.577135,0.582978,0.588401,0.595496,0.600919,0.610511,0.618024,0.624278,0.630951,0.638044,0.644295,0.652217,0.658468,0.665549,0.669711,0.672195,0.672166,0.666279,0.658313,0.652444,0.644068,0.634845,0.623947,0.613037,0.605468,0.6008,0.600317,0.59984,0.598095,0.595123,0.587955,0.577876,0.569483,0.558163,0.548515,0.537615,0.527547,0.519159,0.51243,0.507384,0.50399,0.502676,0.50011,0.495895,0.490852,0.482063,0.478716]
      self.lookup_table_eta_stack = OneDimensionalLookupTable(self.data_power, self.data_eta_stack)
      self.data_power2 = [0.000711683,0.007193702,0.01628193,0.027991228,0.042321579,0.057973684,0.073655526,0.095878947,0.112897368,0.131237719,0.154842105,0.178454386,0.199452632,0.22175,0.245377193,0.267689474,0.291286842,0.314899123,0.335882456,0.359464912,0.383046491,0.404007895,0.424969298,0.447229825,0.469482456,0.490436842,0.511382456,0.529722807,0.550676316,0.571645614,0.59260614,0.613575439,0.635865789,0.659470175,0.681775439,0.702773684,0.723778947,0.744792982,0.76580614,0.786819298,0.807809649,0.831429825,0.855026316,0.875987719]
      self.data_eta_sys = [0.61072,0.616026,0.622089,0.627389,0.631928,0.634947,0.63493,0.634147,0.631092,0.626518,0.618144,0.60901,0.599879,0.591506,0.580854,0.570963,0.563347,0.554213,0.546601,0.540503,0.534405,0.52907,0.523734,0.519156,0.515337,0.51076,0.506942,0.502368,0.497791,0.491697,0.486361,0.480266,0.472652,0.464277,0.455145,0.446014,0.436124,0.425475,0.414826,0.404177,0.395806,0.385913,0.378297,0.372961]
      self.lookup_table_eta_sys = OneDimensionalLookupTable(self.data_power2, self.data_eta_sys)

      self.data_SOC = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
      self.data_A = [-0.001423138720275975, -0.0018802255945328348, -0.00203007574097413, -0.002075654624649823, -0.002107267422887427, -0.00215347204032747, -0.002207351460939956, -0.002249895133354562, -0.0022727808354729563, -0.0023013759276595616]
      self.lookup_table_A_bat = OneDimensionalLookupTable(self.data_SOC , self.data_A)
      self.data_B = [0.04426826532092607, -0.012759286374872602, -0.026458650363584232, -0.031615641550819945, -0.03455608382910607, -0.03689570823278047, -0.03860557660727962, -0.039239310491902636, -0.03867682980713066, -0.0377977700863863]
      self.lookup_table_B_bat = OneDimensionalLookupTable(self.data_SOC , self.data_B)

  def power_req_calculation(self,Vel, Acc, theta = 0):

    cos_theta = math.cos(math.radians(theta))
    if Vel>0:
      result_rolling_resistance=self.mass_vehicle*self.g*self.f_r*cos_theta
    else:
      result_rolling_resistance=0

    result_aerodynamic_resistance = 0.5*self.rho*self.c_d*self.A_f*Vel**2

    sin_theta = math.sin(math.radians(theta))
    gravity_force=self.mass_vehicle*self.g*sin_theta

    result_inertia_force=self.mass_vehicle*Acc

    result_tractiv_force = result_rolling_resistance + gravity_force + result_aerodynamic_resistance + result_inertia_force

    result_tractiv_torqe=result_tractiv_force*self.R_wheel

    result_omega_wheel = Vel/self.R_wheel
    result_omega_dot_wheel = Acc/self.R_wheel
    result_torqe_wheel = 4*self.J_wheel*result_omega_dot_wheel+result_tractiv_torqe

    result_omega_EM=result_omega_wheel*self.GR
    result_omega_dot_EM=result_omega_dot_wheel*self.GR
    if result_torqe_wheel >= 0:
      result_torqe_gb=result_torqe_wheel/(self.eta_gb*self.GR)
    else:
      result_torqe_gb=(result_torqe_wheel*self.eta_gb)/self.GR
    result_power_gb=result_torqe_gb*result_omega_EM

    result_max_torqe_EM=self.lookup_table_max_EM_torqe.lookup((result_omega_EM*60/(2*math.pi)))
    result_1=(self.J_em*result_omega_dot_EM)+result_torqe_gb

    if result_1 > result_max_torqe_EM:
      result_torqe_EM = result_max_torqe_EM
    elif -result_max_torqe_EM <= result_1 and result_1 <= result_max_torqe_EM:
      result_torqe_EM = result_1
    else :
      result_torqe_EM =-result_max_torqe_EM

    result_power_EM=result_torqe_EM*result_omega_EM

    result_eff_EM=float(self.interp_function(result_omega_EM,result_torqe_EM))

    if result_torqe_EM >= 0:
      power_req_new =((result_power_EM/result_eff_EM)+self.P_axu)*0.001
    else:
      power_req_new =((result_power_EM*result_eff_EM)+self.P_axu)*0.001

    if power_req_new > self.lookup_table_max_EM_power.lookup(Vel):
      power_req_new = self.lookup_table_max_EM_power.lookup(Vel)

    return power_req_new

  def calculate_power(self, P_req, power_fc_last, SOC_in):
      if P_req < 0:
          P_fc = 0
          P_bat = P_req
      elif P_req == 0:
          P_fc = 0
          P_bat = 0
      elif 0 < P_req < 7.000 and SOC_in >= 57:
          P_fc = 0
          P_bat = P_req
      elif 0 < P_req < 7.000 and 50 < SOC_in < 60 and power_fc_last == 0:
          P_fc = 0
          P_bat = P_req
      elif 0 < P_req < 7.000 and SOC_in < 57:
          P_fc = 7.000
          P_bat = P_req - 7.000
      elif 7.000 <= P_req < 11.000:
          P_bat = 0.440
          P_fc = P_req - 0.440
      elif 11.000 <= P_req and SOC_in < 57:
          P_bat = 0
          P_fc = P_req
      elif SOC_in >= 57 and 11.000 <= P_req < 14.000:
          P_fc = -0.79 * P_req + 19.11
          P_bat = P_req - P_fc
      elif SOC_in >= 57 and 14.000 <= P_req < 16.500:
          P_fc = 0.5 * P_req + 0.925
          P_bat = P_req - P_fc
      elif SOC_in >= 57 and 16.500 <= P_req < 20.700:
          P_fc = 0.6265 * P_req - 1.168
          P_bat = P_req - P_fc
      elif SOC_in >= 57 and 20.700 <= P_req < 23.500:
          P_fc = 0.92 * P_req - 7.2
          P_bat = P_req - P_fc
      elif SOC_in >= 57 and 23.500 <= P_req:
          P_fc = 0.89 * P_req - 7.7285
          P_bat = P_req - P_fc
      return float(P_bat), float(P_fc)

  def delta_T_bat_calc (self,T_b_s, T_b_c , I_bat,R_bat,T_b_coolant= 298.15, R_b_ext=1.052,R_b_int=0.457,C_b_s=791.19,C_b_c=50.07 ):
    P_bat_loss=R_bat*I_bat**2
    delta_T_b_s=((T_b_coolant-T_b_s)/(R_b_ext*C_b_s))-((T_b_s-T_b_c)/(R_b_int*C_b_s))
    delta_T_b_c = ((T_b_s-T_b_c)/(R_b_int*C_b_c))+((P_bat_loss)/(C_b_c))
    return delta_T_b_s, delta_T_b_c

  def reward_calculation (self, step, power_fc_last, power_req_new, SOC, SOH_bat, SOH_fc,T_bat, power_fc, power_bat):

    power_fc *= .95**2
    power_bat *= .98

    if power_bat >= 0:
      result_P_bat = power_bat*1000
      P_disc = result_P_bat
      P_cha=0
    else:
      result_P_bat=power_bat*(-1000)
      P_disc = 0
      P_cha = result_P_bat

    result_OCV=1722.4*((SOC)**5)-4747.5*((SOC)**4)+4891.9*((SOC)**3)-2312.1*((SOC)**2)+525.72*(SOC)+200.06
    result_R_disc=0.0188*((SOC)**4)-0.0547*((SOC)**3)+0.0765*((SOC)**2)-0.0457*((SOC))+0.0349
    result_R_cha=0.0056*((SOC)**4)-0.0254*((SOC)**3)+0.0372*((SOC)**2)-0.0203*((SOC))+0.0223
    result_I_disc=(result_OCV-math.sqrt((result_OCV**2)-4*result_R_disc*P_disc))/(2*result_R_disc)
    result_I_cha=(-result_OCV+math.sqrt((result_OCV**2)+4*result_R_cha *P_cha))/(2*result_R_cha)
    result_deltaSOC=((result_I_cha*self.time_step)/self.Q_nom)-(((result_I_disc**1.1)*self.time_step)/self.Q_nom)
    result_SOC_new=((result_deltaSOC)+SOC)

    if result_SOC_new > 1:
      result_SOC_new = 1
    if result_SOC_new < 0:
      result_SOC_new = 0

    result_I_bat=result_I_disc-result_I_cha
    result_C_rate=(abs((result_I_bat)/(self.Q_nom/3600)))
    M=self.lookup_table_M.lookup(result_C_rate)
    result_Q_loss=M*math.exp((-31700+(370.3*result_C_rate))/(8.31*(T_bat)))
    result_Ah_tot=(20/(result_Q_loss))**0.55
    result_deltaSOH=-(((abs(result_I_bat)*self.time_step)/(result_Ah_tot*3600*2))) * 0.01

    result_eta_stack = self.lookup_table_eta_stack.lookup((power_fc/self.P_max_fc))
    result_eta_sys = self.lookup_table_eta_sys.lookup((power_fc/self.P_max_fc))

    power_th = (result_eta_stack/result_eta_sys)*power_fc
    result_Hydrogen_flow_rate = 8.5e-5*(power_th**2)+9.1e-3*(power_th)+0.0064
    if power_fc < 0:
      result_Hydrogen_flow_rate = 0

    deltaP = abs(power_fc - power_fc_last)
    result_P_rel = power_fc/self.P_max_fc
    if result_P_rel >= 0.8:
      result_High_load_degradation_rate = (self.time_step/3600)*10.0*10**(-6)
    else:
      result_High_load_degradation_rate = 0
    if result_P_rel <= 0.2 and result_P_rel > 0 :
      result_Low_load_degradation_rate = (self.time_step/3600)*8.662*10**(-6)
    else:
      result_Low_load_degradation_rate = 0
    if step != 0 :
      result_Load_difference_degradation_rate = 4.185*10**(-8)*deltaP/self.P_max_fc
    else:
      result_Load_difference_degradation_rate=0
    if power_fc != 0 and power_fc_last == 0:
      result_start_stop_degradation_rate = 13.79*1E-8
    else:
      result_start_stop_degradation_rate=0

    result_totall_degradation_rate = result_High_load_degradation_rate + result_Low_load_degradation_rate + result_Load_difference_degradation_rate + result_start_stop_degradation_rate
    delta_V_d=result_totall_degradation_rate
    #update state
    power_fc_last = power_fc
    SOC = result_SOC_new
    result_SOH_new = result_deltaSOH + SOH_bat
    SOH_fc_new = SOH_fc - (delta_V_d/0.07)
    A=self.lookup_table_A_bat.lookup(SOC)
    B=self.lookup_table_B_bat.lookup(SOC)
    #cost calculation
    m_equ = A * (result_deltaSOC*self.Q_nom) + B
    if m_equ < 0:
      m_equ = 0
    C_H2 = 32.94 * 0.001 * (result_Hydrogen_flow_rate + m_equ)
    C_bat = self.Q_bat * 139 * abs(result_deltaSOH)
    C_fc = self.P_max_fc * 93 * (delta_V_d/0.07)
    C_soc = 1.5 * (0.575 - SOC)**2 

    cost_real =  (C_H2+C_bat+C_fc)
    reward = - (cost_real + C_soc )

    out={}
    out['power_fc_last'] = power_fc_last
    out['SOC'] = SOC
    out['SOH_bat'] = result_SOH_new
    out['SOH_fc'] = SOH_fc_new
    out['m_dot_hydrohen'] = result_Hydrogen_flow_rate
    out['m_equ'] = m_equ
    out['A'] = float(A)
    out['B'] = float(B)
    out['result_deltaSOH'] = result_deltaSOH
    out['result_totall_degradation_rate'] = result_totall_degradation_rate
    out['deltaSOC'] = result_deltaSOC
    out['R_disch'] = result_R_disc
    out['R_cha'] = result_R_cha
    out['OCV'] = result_OCV
    out['I_disch'] = result_I_disc
    out['I_cha'] = result_I_cha
    out['deltaP'] = float(deltaP)
    out['result_P_rel'] =  float(result_P_rel)
    out['C_H2'] = C_H2
    out['C_fc'] = C_fc
    out['C_bat'] = C_bat
    out['cost_real'] = cost_real

    return out, reward

**Define actor & critic networks class**

In [ ]:
BUFFER_SIZE = int(1e4)        # replay buffer size
BATCH_SIZE = 256              # minibatch size
GAMMA = 0.9                   # discount factor
TAU = 1e-2                    # for soft update of target parameters
LR_ACTOR = .5e-3              # learning rate of the actor
LR_CRITIC = .5e-3             # learning rate of the critic
WEIGHT_DECAY = 0#1e-2         # L2 weight decay

GPU = True
device_idx = 0
if GPU:
    device = torch.device("cuda:" + str(device_idx) if torch.cuda.is_available() else "cpu")
else:
    device = torch.device("cpu")
print(device)

def hidden_init(layer):
    fan_in = layer.weight.data.size()[0]
    lim = 1. / np.sqrt(fan_in)
    return (-lim, lim)

class Actor(nn.Module):
    def __init__(self, state_size, action_size, seed, hidden_size=256, init_w=3e-3, log_std_min=-20, log_std_max=1.5):

        super(Actor, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max

        self.fc1 = nn.Linear(state_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, hidden_size)

        self.mu = nn.Linear(hidden_size, action_size)
        self.log_std_linear = nn.Linear(hidden_size, action_size)

    def reset_parameters(self):
        self.fc1.weight.data.uniform_(*hidden_init(self.fc1))
        self.fc2.weight.data.uniform_(*hidden_init(self.fc2))
        self.mu.weight.data.uniform_(-init_w, init_w)
        self.log_std_linear.weight.data.uniform_(-init_w, init_w)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        mu = self.mu(x)
        log_std = self.log_std_linear(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mu, log_std

    def evaluate(self, state, epsilon=1e-6, var=1):
        mu, log_std = self.forward(state)
        std = log_std.exp()

        dist = Normal(0, 3)
        e = dist.sample().to(device)

        action = torch.tanh(mu + e * std)
        log_prob = Normal(mu, std).log_prob(mu + e * std) - torch.log(1 - action.pow(2) + epsilon)
        return action, log_prob


    def get_action(self, state):

        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        mu, log_std = self.forward(state)
        std = log_std.exp()

        dist = Normal(0, 3)
        e = dist.sample().to(device)
        action = torch.tanh(mu + e * std).cpu()
        return action[0]


class Critic(nn.Module):

    def __init__(self, state_size, action_size, seed, hidden_size=256):

        super(Critic, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size+action_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.fc4 = nn.Linear(hidden_size, 1)
        self.reset_parameters()

    def reset_parameters(self):
        self.fc1.weight.data.uniform_(*hidden_init(self.fc1))
        self.fc2.weight.data.uniform_(*hidden_init(self.fc2))
        self.fc3.weight.data.uniform_(-3e-3, 3e-3)

    def forward(self, state, action):
        x = torch.cat((state, action), dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        return self.fc4(x)


**Define soft actor-crittic learning agent class**

In [ ]:
class Agent():
    def __init__(self, state_size, action_size, random_seed, action_prior="uniform"):

        self.state_size = state_size
        self.action_size = action_size
        self.seed = random.seed(random_seed)

        self.target_entropy = -action_size  # -dim(A)
        self.alpha = 1
        self.log_alpha = torch.tensor([0.0], requires_grad=True, device=device)
        self.alpha_optimizer = optim.Adam(params=[self.log_alpha], lr=LR_ACTOR)
        self._action_prior = action_prior

        print("Using: ", device)

        # Actor Network
        self.actor_local = Actor(state_size, action_size, random_seed).to(device)
        self.actor_optimizer = optim.Adam(self.actor_local.parameters(), lr=LR_ACTOR)

        # Critic Network (w/ Target Network)
        self.critic1 = Critic(state_size, action_size, random_seed).to(device)
        self.critic2 = Critic(state_size, action_size, random_seed).to(device)

        self.critic1_target = Critic(state_size, action_size, random_seed).to(device)
        self.critic1_target.load_state_dict(self.critic1.state_dict())

        self.critic2_target = Critic(state_size, action_size, random_seed).to(device)
        self.critic2_target.load_state_dict(self.critic2.state_dict())

        self.critic1_optimizer = optim.Adam(self.critic1.parameters(), lr=LR_CRITIC, weight_decay=WEIGHT_DECAY)
        self.critic2_optimizer = optim.Adam(self.critic2.parameters(), lr=LR_CRITIC, weight_decay=WEIGHT_DECAY)

        # Replay memory
        self.memory = ReplayBuffer(action_size, BUFFER_SIZE, BATCH_SIZE, random_seed)

    def step(self, state, action, reward, next_state, done, step, var):
        """Save experience in replay memory, and use random sample from buffer to learn."""
        # Save experience / reward
        self.memory.add(state, action, reward, next_state, done)

        # Learn, if enough samples are available in memory
        if len(self.memory) > BATCH_SIZE:
            experiences = self.memory.sample()
            self.learn(step, experiences, GAMMA, var=var)


    def act(self, state, add_noise=True, var=1):
        """Returns actions for given state as per current policy."""
        state = torch.from_numpy(state).unsqueeze(0).float().to(device)
        action, _ = self.actor_local.evaluate(state, var)
        return action

    def learn(self, step, experiences, gamma, d=1, var=1):

        states, actions, rewards, next_states, dones = experiences
        # ---------------------------- update critic ---------------------------- #
        next_action, log_pis_next = self.actor_local.evaluate(next_states)

        Q_target1_next = self.critic1_target(next_states.to(device), next_action.squeeze(0).to(device))
        Q_target2_next = self.critic2_target(next_states.to(device), next_action.squeeze(0).to(device))

        Q_target_next = torch.min(Q_target1_next, Q_target2_next)

        Q_targets = rewards + (gamma * (1 - dones) * (Q_target_next - self.alpha * log_pis_next.squeeze(0)))

        Q_1 = self.critic1(states, actions)
        Q_2 = self.critic2(states, actions)
        critic1_loss = 0.5*F.mse_loss(Q_1, Q_targets.detach())
        critic2_loss = 0.5*F.mse_loss(Q_2, Q_targets.detach())

        self.critic1_optimizer.zero_grad()
        critic1_loss.backward()
        self.critic1_optimizer.step()

        self.critic2_optimizer.zero_grad()
        critic2_loss.backward()
        self.critic2_optimizer.step()
        if step % d == 0:
        # ---------------------------- update actor ---------------------------- #
            alpha = torch.exp(self.log_alpha)
            actions_pred, log_pis = self.actor_local.evaluate(states, var)
            alpha_loss = - (self.log_alpha * (log_pis + self.target_entropy).detach()).mean()
            self.alpha_optimizer.zero_grad()
            alpha_loss.backward()
            self.alpha_optimizer.step()

            self.alpha = alpha
            if self._action_prior == "normal":
                policy_prior = MultivariateNormal(loc=torch.zeros(self.action_size), scale_tril=torch.ones(self.action_size).unsqueeze(0))
                policy_prior_log_probs = policy_prior.log_prob(actions_pred)
            elif self._action_prior == "uniform":
                policy_prior_log_probs = 0.0

            actor_loss = (alpha * log_pis.squeeze(0) - self.critic1(states, actions_pred.squeeze(0)) - policy_prior_log_probs ).mean()

            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()

            # ----------------------- update target networks ----------------------- #
            self.soft_update(self.critic1, self.critic1_target, TAU)
            self.soft_update(self.critic2, self.critic2_target, TAU)

    def soft_update(self, local_model, target_model, tau):
        for target_param, local_param in zip(target_model.parameters(), local_model.parameters()):
            target_param.data.copy_(tau*local_param.data + (1.0-tau)*target_param.data)

**Define Replay Buffer class**

In [ ]:
class ReplayBuffer:

    def __init__(self, action_size, buffer_size, batch_size, seed):

        self.action_size = action_size
        self.memory = deque(maxlen=buffer_size)  # internal memory (deque)
        self.batch_size = batch_size
        self.experience = namedtuple("Experience", field_names=["state", "action", "reward", "next_state", "done"])
        self.seed = random.seed(seed)

    def add(self, state, action, reward, next_state, done):
        e = self.experience(state, action, reward, next_state, done)
        self.memory.append(e)

    def sample(self):
        experiences = random.sample(self.memory, k=self.batch_size)

        states = torch.from_numpy(np.vstack([e.state for e in experiences if e is not None])).float().to(device)
        actions = torch.from_numpy(np.vstack([e.action for e in experiences if e is not None])).float().to(device)
        rewards = torch.from_numpy(np.vstack([e.reward for e in experiences if e is not None])).float().to(device)
        next_states = torch.from_numpy(np.vstack([e.next_state for e in experiences if e is not None])).float().to(device)
        dones = torch.from_numpy(np.vstack([e.done for e in experiences if e is not None]).astype(np.uint8)).float().to(device)

        return (states, actions, rewards, next_states, dones)

    def __len__(self):
        return len(self.memory)

In [ ]:
if __name__ == '__main__':

    bat_size = 1.6 #kWh
    C_rate_max = 14
    alpha_power = 1.4
    power_bat_max = C_rate_max*bat_size
    fc_size = 114.5 #kW
    
    data_path = r"\...\...\Standard_UDDS.mat"
    data = scio.loadmat(data_path)
    car_spd_one = data['speed_vector'] # mps
    total_dis = np.sum(car_spd_one) / 1000   #distance in km

    rand_seed = 1
    while True:

        break_while = True
        cost_real_list = []
        total_step = 0
        step_episode = 0
        mean_reward_all = 0
        cost_Engine_list = []
        fuel_consumption_100km_list=[]
        fuel_consumptin_list = []
        reward_all_list = []
        cost_Engine_100Km_list = []
        mean_reward_list = []
        mean_discrepancy_list = []
        SOC_final_list = []
        Avg_socre_list=[]
        score_history = []
        delta_SOC_list=[]
        m_dot_equ=[]
        R_disch=[]
        R_cha=[]
        OCV=[]
        I_disch=[]
        I_cha=[]
        s_dim = [2]
        a_dim = 1
        Mirai = Mirai_model_V2(P_max_fc = fc_size, Q_bat = bat_size)
        env =  Mirai
        env =  Mirai
        MAX_EPISODES = 150
        best_cost= float('inf')

        var = 2
        state_size = (2)
        action_size = 1
        agent = Agent(state_size=state_size, action_size=action_size, random_seed=rand_seed, action_prior="uniform") #"normal"

        for i in range(MAX_EPISODES):

            done = False
            SOC = 0.6
            SOC_origin = SOC
            power_fc_last=0
            power_fc_last_origin=power_fc_last
            power_req=0.440
            power_req_origin=power_req
            SOH_bat=1
            SOH_bat_origin=SOH_bat
            SOH_fc=1
            SOH_fc_origin=SOH_fc
            score = 0
            cost_all=0
            ep_reward_all = 0
            step_episode += 1
            fuel_consumptin=0
            total_cost=0
            cost_real=0
            SOC_data = []
            power_fc_last_list=[]
            SOC_list=[]
            power_req_list=[]
            power_bat_list=[]
            SOH_bat_list=[]
            SOH_fc_list=[]
            result_Hydrogen_flow_rate_list=[]
            Reward_list = []
            action_list = []

            car_spd = car_spd_one[:, 0]
            car_a = car_spd_one[:, 0] - 0
            power_req_new = Mirai.power_req_calculation(car_spd, car_a)
            s = np.zeros(s_dim)
            s[0] = power_req_new
            s[1] = SOC_origin

            for j in range(car_spd_one.shape[1] - 1):

                action = agent.act(s, var)
                action = float(action[0])
                power_fc = (fc_size*((action+1)/2))
                
                if power_fc > fc_size:
                    power_fc=fc_size
                if power_fc < 0:
                    powet_fc=0

                power_bat = s[0]-power_fc
                if power_bat > power_bat_max:
                    power_bat = power_bat_max
                    power_fc = s[0]-power_bat

                if power_bat < -power_bat_max:
                    power_bat = -power_bat_max

                power_bat_list.append((power_bat))
                
                out, reward = Mirai.reward_calculation(j,power_fc_last, s[0], s[1], SOH_bat, SOH_fc, power_fc, power_bat)

                power_fc_last_list.append(float(out['power_fc_last']))
                SOC_list.append(float(out['SOC']))
                delta_SOC_list.append(float(out['deltaSOC']))
                SOH_bat_list.append(float(out['SOH_bat']))
                SOH_fc_list.append(float(out['SOH_fc']))
                result_Hydrogen_flow_rate_list.append(float(out['m_dot_hydrohen']))
                R_disch.append(float(out['R_disch']))
                R_cha.append(float(out['R_cha']))
                OCV.append(float(out['OCV']))
                I_disch.append(float(out['I_disch']))
                I_cha.append(float(out['I_cha']))
                power_fc_last_new=float(out['power_fc_last'])
                SOC_new=float(out['SOC'])
                SOH_bat_new=float(out['SOH_bat'])
                SOH_fc_new=float(out['SOH_fc'])
                result_Hydrogen_flow_rate=float(out['m_dot_hydrohen'])
                deltaSOC=float(out['deltaSOC'])

                score += reward
                Reward_list.append(reward)
                cost_real += float(out['cost_real'])


                fuel_consumptin +=result_Hydrogen_flow_rate
                fuel_consumptin_list.append(fuel_consumptin)
                
                # Obtained from the wheel speed sensor
                car_spd = car_spd_one[:, j + 1]
                car_a = car_spd_one[:, j + 1] - car_spd_one[:, j]
                power_req_new = Mirai.power_req_calculation(car_spd, car_a)
                power_req_list.append(power_req_new)

                s_ = np.zeros(s_dim)
                s_[0] = power_req_new
                s_[1] = SOC_new

                agent.step(s, action, reward, s_, done, j, var)
                s = s_
                power_fc_last = power_fc_last_new
                SOC = SOC_new
                SOH_bat = SOH_bat_new
                SOH_fc = SOH_fc_new
                total_step += 1


            if j == (car_spd_one.shape[1] - 2):
                SOC_final_list.append(SOC)
                score += ((float(out['A'])) * (SOC - SOC_origin) * ((bat_size*1000*3600)/245) + float(out['B'])) * 0.001 * 32.94
                mean_reward = score / car_spd_one.shape[1]

                mean_reward_list.append(mean_reward)
                score_history.append(score)
                if step_episode >= 100:
                    Avg_socre = sum(score_history[-100:]) / 100
                else:
                    Avg_socre = sum(score_history[-step_episode:]) / step_episode
                Avg_socre_list.append(Avg_socre)
                fuel_consumption_100km_list.append(((fuel_consumptin)/total_dis)*100)
                print('Episode:', i, ' Total score: %.3f' % score,' cost real: %.3f' % cost_real,' ep_fuel_cons: %.3f' % fuel_consumptin, ' mean_socre_last_100EP: %.3f' % Avg_socre,' SOC-final: %.3f' % SOC,' SOH_bat_final: %.3f' % (SOH_bat*100),' SOH_fc_final: %.3f' % (SOH_fc*100))

            if best_cost > cost_real:
                print("--------saving-------")
                torch.save(agent.actor_local.state_dict(), dir+r'/final_actor.h5')
                torch.save(agent.critic1.state_dict(), dir+r'/final_critic1.h5')
                torch.save(agent.critic2.state_dict(), dir+r'/final_critic2.h5')
                best_cost = cost_real
                SOC_last_episod = SOC_list
                SOH_bat_last_episod = SOH_bat_list
                SOH_fc_last_episod = SOH_fc_list
                best_fuel_consumptin = fuel_consumptin
                power_req_episod = power_req_list
                power_bat_episod = power_bat_list
                power_fc_last_episod = power_fc_last_list
                fuel_consumptin_100km = fuel_consumption_100km_list
                fuel_consumptin_list_best=fuel_consumptin_list
                cost_real_best = cost_real

            cost_real_list.append(cost_real)
            if i > 4:
                if (round(cost_real_list[-4], 3) == round(cost_real_list[-3], 3) == 
                    round(cost_real_list[-2], 3) == round(cost_real_list[-1], 3)) and score<-200 :
                    print("failed")
                    break_while = False
                    break

        rand_seed +=1
        if break_while:
            break

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plotLearning(scores, filename, x=None, window=5):
    N = len(scores)
    running_avg = np.empty(N)
    for t in range(N):
	    running_avg[t] = np.mean(scores[max(0, t-window):(t+1)])
    if x is None:
        x = [i for i in range(N)]
    plt.ylabel('Score')
    plt.xlabel('Episode')
    plt.plot(x, running_avg)
    plt.savefig(filename)

filename = 'scores of Episodes.png'
plotLearning(score_history, filename, window=1)

In [ ]:
COST_TOTAL = float((200000/total_dis*cost_real_best) + (fc_size*93) + (bat_size*139)) #Total cost for 200,000 km
print(COST_TOTAL)

data_SAC = {'cost_total':COST_TOTAL,'score':score_history,'fuel_comolative' :fuel_consumptin_list_best,'fuel_100km' :fuel_consumption_100km_list, 'SOC_best_ep':SOC_last_episod,'power_bat':power_bat_list,'power_fc':power_fc_last,'SOH_bat':SOH_bat_last_episod,'SOH_fc':SOH_fc_last_episod, "cost_history":cost_history}
scio.savemat(dir+r'\data_SAC.mat',data_SAC)